# 04 — Ensemble demand forecast with FIFO production policy

Train three demand models with rolling chronological validation, select ensemble weights, test FIFO production buffers, and produce the next-day recommendation.

Costs are never demand features. They are used only after forecasting to compare stockouts with expired stock. Actual results, when available, are joined only after model training for evaluation.

In [ ]:
from pathlib import Path
import os
import sys
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

CURRENT_FOLDER = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        folder
        for folder in [CURRENT_FOLDER, *CURRENT_FOLDER.parents]
        if (folder / "src").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError("Could not find the pastry-sales-forecasting project folder.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.calendar_features import create_calendar
from src.demand_model import (
    CATEGORICAL_DEFAULTS,
    NUMERIC_DEFAULTS,
    LAGS,
    build_features,
    calculate_metrics,
    make_model_specs,
    make_pipeline,
    make_rolling_folds,
    select_model_features,
)
from src.inventory_simulation import (
    add_business_costs,
    simulate_fifo_strategy,
    summarize_fifo_results,
)

DATA_MODE = os.getenv("PASTRY_DATA_MODE", "synthetic").strip().lower()
if DATA_MODE not in {"synthetic", "private"}:
    raise ValueError("PASTRY_DATA_MODE must be 'synthetic' or 'private'.")

if DATA_MODE == "private":
    PREPARED_FILE = PROJECT_ROOT / "data" / "private" / "processed" / "production.pkl"
    EVALUATION_FILE = PROJECT_ROOT / "data" / "private" / "processed" / "production_evaluation.pkl"
    OUTPUTS_PATH = PROJECT_ROOT / "outputs" / "private"
else:
    PREPARED_FILE = PROJECT_ROOT / "data" / "processed" / "production.pkl"
    EVALUATION_FILE = PROJECT_ROOT / "data" / "processed" / "production_evaluation.pkl"
    OUTPUTS_PATH = PROJECT_ROOT / "outputs"

OUTPUTS_PATH.mkdir(parents=True, exist_ok=True)

print("Data mode:", DATA_MODE)
print("Prepared model file:", PREPARED_FILE)
print("Optional evaluation file:", EVALUATION_FILE)


## Load the prepared demand target

In [ ]:
if not PREPARED_FILE.exists():
    raise FileNotFoundError(
        f"Prepared data was not found: {PREPARED_FILE}\n"
        "Run notebooks/01_data_preparation.ipynb first."
    )

production = pd.read_pickle(PREPARED_FILE)
production["Date"] = pd.to_datetime(production["Date"]).dt.normalize()

# Fixed historical cutoff: August 1 is forecast/evaluation only.
TRAINING_END_DATE = pd.Timestamp("2026-07-31")
FORECAST_DATE = pd.Timestamp("2026-08-01")

required_columns = {
    "Date",
    "Store",
    "Product",
    "Sales",
    "UseForSalesModel",
    "ClosingStock",
    "CarryoverStock",
    "ListedPrice",
    "UnitCost",
    "UnitMarginEstimate",
}
missing_columns = required_columns.difference(production.columns)
if missing_columns:
    raise KeyError(f"Missing required columns: {sorted(missing_columns)}")

base_data = production.loc[
    production["UseForSalesModel"].fillna(False)
    & production["Date"].le(TRAINING_END_DATE)
].copy()
base_data["Demand"] = pd.to_numeric(base_data["Sales"], errors="coerce")
base_data = base_data.loc[
    base_data["Demand"].notna() & base_data["Demand"].ge(0)
].copy()
base_data = base_data.sort_values(
    ["Date", "Store", "Product"]
).reset_index(drop=True)

if base_data["Date"].max() > TRAINING_END_DATE:
    raise AssertionError("Training data contains dates after July 31, 2026.")

print("Training cutoff:", TRAINING_END_DATE.date())
print("Forecast date:", FORECAST_DATE.date())
print("Rows:", len(base_data))
print("Stores:", base_data["Store"].nunique())
print("Products:", base_data["Product"].nunique())
print("Date range:", base_data["Date"].min(), "to", base_data["Date"].max())
print(
    "Historical weather rows:",
    int(base_data.get("TemperatureAvg", pd.Series(index=base_data.index, dtype=float)).notna().sum()),
)


In [ ]:
display(base_data.head())


## Build leakage-safe features and validation folds

In [ ]:
ORIGIN_DATE = base_data["Date"].min()
model_data = build_features(base_data, ORIGIN_DATE)
CATEGORICAL_FEATURES, NUMERIC_FEATURES, MODEL_FEATURES = select_model_features(model_data)
TARGET = "Demand"

for column in NUMERIC_FEATURES:
    model_data[column] = pd.to_numeric(
        model_data[column], errors="coerce"
    ).astype(float)

print("Model features:", len(MODEL_FEATURES))
print("Weather used:", "TemperatureAvg" in NUMERIC_FEATURES)


In [ ]:
display(
    model_data[
        ["Date", "Store", "Product", "Demand", "Lag7", "Recent7Mean", "Weather"]
    ].head(15)
)


In [ ]:
folds = make_rolling_folds(
    model_data,
    fold_days=7,
    max_folds=4,
)

print("Rolling validation folds:")
for fold_start, fold_end, validation_dates in folds:
    print(
        f"  {fold_start.date()} to {fold_end.date()} "
        f"({len(validation_dates)} dates)"
    )


## Train rolling-validation models

In [ ]:
model_specs = make_model_specs()
oof_frames = []

for fold_number, (fold_start, fold_end, validation_dates) in enumerate(folds, start=1):
    train_data = model_data.loc[model_data["Date"].lt(fold_start)].copy()
    validation_data = model_data.loc[
        model_data["Date"].isin(validation_dates)
        & model_data["Lag7"].notna()
    ].copy()

    if train_data.empty or validation_data.empty:
        continue

    fold_predictions = validation_data[
        [
            "Date",
            "Store",
            "Product",
            "Type",
            "Demand",
            "CarryoverStock",
            "ListedPrice",
            "UnitCost",
            "UnitMarginEstimate",
            "Recent7Mean",
            "Lag7",
        ]
    ].copy()
    fold_predictions["Fold"] = fold_number

    for model_name, estimator in model_specs.items():
        pipeline = make_pipeline(
            estimator,
            CATEGORICAL_FEATURES,
            NUMERIC_FEATURES,
        )
        pipeline.fit(train_data[MODEL_FEATURES], train_data[TARGET])
        fold_predictions[model_name] = np.clip(
            pipeline.predict(validation_data[MODEL_FEATURES]),
            0,
            None,
        )

    oof_frames.append(fold_predictions)
    print(
        f"Fold {fold_number}: train rows={len(train_data)}, "
        f"validation rows={len(validation_data)}"
    )

oof_predictions = pd.concat(oof_frames, ignore_index=True)
print("Out-of-fold rows:", len(oof_predictions))


## Select ensemble weights

In [ ]:
MODEL_NAMES = list(model_specs)
weight_rows = []
weight_values = np.arange(0, 1.01, 0.05)

for first_weight in weight_values:
    for second_weight in weight_values:
        third_weight = 1 - first_weight - second_weight
        if third_weight < -1e-9:
            continue

        raw_prediction = (
            first_weight * oof_predictions[MODEL_NAMES[0]]
            + second_weight * oof_predictions[MODEL_NAMES[1]]
            + third_weight * oof_predictions[MODEL_NAMES[2]]
        )
        rounded_prediction = np.clip(np.rint(raw_prediction), 0, None)
        metrics = calculate_metrics(
            oof_predictions["Demand"],
            rounded_prediction,
        )
        weight_rows.append(
            {
                MODEL_NAMES[0]: first_weight,
                MODEL_NAMES[1]: second_weight,
                MODEL_NAMES[2]: third_weight,
                **metrics,
            }
        )

weight_results = pd.DataFrame(weight_rows).sort_values(
    ["MAE", "RMSE", "Bias"],
    key=lambda values: values.abs() if values.name == "Bias" else values,
).reset_index(drop=True)

best_weights = weight_results.iloc[0]
ENSEMBLE_WEIGHTS = {
    model_name: float(best_weights[model_name])
    for model_name in MODEL_NAMES
}

print("Selected ensemble weights:")
for model_name, weight in ENSEMBLE_WEIGHTS.items():
    print(f"  {model_name}: {weight:.2f}")

oof_predictions["EnsembleRaw"] = sum(
    ENSEMBLE_WEIGHTS[model_name] * oof_predictions[model_name]
    for model_name in MODEL_NAMES
)
oof_predictions["EnsemblePrediction"] = np.clip(
    np.rint(oof_predictions["EnsembleRaw"]),
    0,
    None,
)
oof_predictions["Residual"] = (
    oof_predictions["Demand"] - oof_predictions["EnsembleRaw"]
)


### Validation accuracy

Compare the ensemble with each component model and the two simple historical baselines.

In [ ]:
metric_rows = []
comparison_columns = {
    "Ensemble": "EnsemblePrediction",
    "Recent 7-day mean": "Recent7Mean",
    "Same weekday last week": "Lag7",
    **{model_name: model_name for model_name in MODEL_NAMES},
}

for display_name, prediction_column in comparison_columns.items():
    valid = oof_predictions[["Demand", prediction_column]].dropna()
    prediction = np.clip(np.rint(valid[prediction_column]), 0, None)
    metric_rows.append(
        {
            "Model": display_name,
            **calculate_metrics(valid["Demand"], prediction),
            "Rows": len(valid),
        }
    )

metrics = pd.DataFrame(metric_rows).sort_values("MAE").reset_index(drop=True)
display(metrics.style.format({
    "MAE": "{:.2f}",
    "RMSE": "{:.2f}",
    "WAPE": "{:.1%}",
    "Bias": "{:+.2f}",
}))

baseline_mae = metrics.loc[
    metrics["Model"].eq("Recent 7-day mean"),
    "MAE",
].iloc[0]
ensemble_mae = metrics.loc[
    metrics["Model"].eq("Ensemble"),
    "MAE",
].iloc[0]
print(f"Ensemble improvement versus recent 7-day mean: {(baseline_mae - ensemble_mae) / baseline_mae:.1%}")


### Daily validation totals

Aggregate the rolling-validation predictions by day to check whether total shop demand is tracked.

In [ ]:
daily_comparison = (
    oof_predictions.groupby("Date", as_index=False)
    .agg(
        ActualDemand=("Demand", "sum"),
        PredictedDemand=("EnsemblePrediction", "sum"),
    )
)

plt.figure(figsize=(12, 5))
plt.plot(daily_comparison["Date"], daily_comparison["ActualDemand"], marker="o", label="Actual")
plt.plot(daily_comparison["Date"], daily_comparison["PredictedDemand"], marker="o", label="Ensemble")
plt.title("Rolling validation: total daily demand")
plt.ylabel("Units")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Select a FIFO safety buffer

The demand forecast is tested with buffers from −2 to +10. Old stock sells first; unsold old stock expires; unsold new production becomes tomorrow’s carryover. Product economics select the buffer that best balances fulfilled sales and waste.

In [ ]:
SAFETY_BUFFERS = list(range(-2, 11))
MIN_STRATEGY_ROWS = 3

fifo_input = oof_predictions[
    [
        "Date",
        "Store",
        "Product",
        "Type",
        "Demand",
        "CarryoverStock",
        "ListedPrice",
        "UnitCost",
        "UnitMarginEstimate",
        "EnsembleRaw",
    ]
].copy()

fifo_input = fifo_input.dropna(
    subset=["Demand", "EnsembleRaw"]
).copy()

simulation_frames = []

for safety_buffer in SAFETY_BUFFERS:
    strategy_data = fifo_input.copy()
    strategy_data["SafetyBuffer"] = safety_buffer
    strategy_data["AdjustedForecast"] = np.clip(
        strategy_data["EnsembleRaw"] + safety_buffer,
        0,
        None,
    )

    simulation = simulate_fifo_strategy(
        strategy_data,
        forecast_column="AdjustedForecast",
        demand_column="Demand",
        initial_carryover_column="CarryoverStock",
        model_name="Ensemble",
    )
    simulation = add_business_costs(simulation)
    simulation_frames.append(simulation)

fifo_simulation_details = pd.concat(
    simulation_frames,
    ignore_index=True,
)

fifo_buffer_metrics = summarize_fifo_results(
    fifo_simulation_details,
    group_columns=["SafetyBuffer"],
)
fifo_buffer_metrics["AbsoluteBuffer"] = fifo_buffer_metrics[
    "SafetyBuffer"
].abs()


In [ ]:
eligible_global_buffers = fifo_buffer_metrics.loc[
    fifo_buffer_metrics["CostCoverage"].eq(1.0)
    & fifo_buffer_metrics["Rows"].ge(MIN_STRATEGY_ROWS)
].copy()

if eligible_global_buffers.empty:
    GLOBAL_FIFO_BUFFER = 0
    print(
        "Complete economics were not available for FIFO selection. "
        "Global safety buffer defaults to 0."
    )
else:
    global_choice = (
        eligible_global_buffers
        .sort_values(
            [
                "SalesMinusLossCostYen",
                "StockoutUnits",
                "SimulatedLoss",
                "AbsoluteBuffer",
            ],
            ascending=[False, True, True, True],
        )
        .iloc[0]
    )
    GLOBAL_FIFO_BUFFER = int(global_choice["SafetyBuffer"])


In [ ]:
store_product_metrics = summarize_fifo_results(
    fifo_simulation_details,
    group_columns=["Store", "Product", "SafetyBuffer"],
)
store_product_metrics["AbsoluteBuffer"] = store_product_metrics[
    "SafetyBuffer"
].abs()

eligible_store_product = store_product_metrics.loc[
    store_product_metrics["CostCoverage"].eq(1.0)
    & store_product_metrics["Rows"].ge(MIN_STRATEGY_ROWS)
].copy()

if eligible_store_product.empty:
    selected_fifo_policy = pd.DataFrame(
        columns=[
            "Store",
            "Product",
            "SafetyBuffer",
            "EstimatedSalesMarginYen",
            "EstimatedLossCostYen",
            "EstimatedSalesMinusLossYen",
            "ValidationStockoutUnits",
            "ValidationExpiredUnits",
            "StrategyRows",
        ]
    )
else:
    selected_fifo_policy = (
        eligible_store_product
        .sort_values(
            [
                "Store",
                "Product",
                "SalesMinusLossCostYen",
                "StockoutUnits",
                "SimulatedLoss",
                "AbsoluteBuffer",
            ],
            ascending=[True, True, False, True, True, True],
        )
        .groupby(["Store", "Product"], as_index=False)
        .first()
        .rename(
            columns={
                "SalesMarginYen": "EstimatedSalesMarginYen",
                "LossCostYen": "EstimatedLossCostYen",
                "SalesMinusLossCostYen": "EstimatedSalesMinusLossYen",
                "StockoutUnits": "ValidationStockoutUnits",
                "SimulatedLoss": "ValidationExpiredUnits",
                "Rows": "StrategyRows",
            }
        )
        [
            [
                "Store",
                "Product",
                "SafetyBuffer",
                "EstimatedSalesMarginYen",
                "EstimatedLossCostYen",
                "EstimatedSalesMinusLossYen",
                "ValidationStockoutUnits",
                "ValidationExpiredUnits",
                "StrategyRows",
            ]
        ]
    )

if selected_fifo_policy.empty:
    selected_policy_details = pd.DataFrame()
    selected_policy_summary = pd.DataFrame()
else:
    selected_policy_details = fifo_simulation_details.merge(
        selected_fifo_policy[["Store", "Product", "SafetyBuffer"]],
        on=["Store", "Product", "SafetyBuffer"],
        how="inner",
        validate="many_to_one",
    )
    selected_policy_details["Policy"] = "Ensemble + selected FIFO buffer"
    selected_policy_summary = summarize_fifo_results(
        selected_policy_details,
        group_columns=["Policy"],
    )


In [ ]:
print("Global FIFO safety buffer:", GLOBAL_FIFO_BUFFER)
print("Store–Product FIFO policies:", len(selected_fifo_policy))


### Global buffer comparison

Show the sales, stockout, and expiry trade-off for every tested safety buffer.

In [ ]:
display(
    fifo_buffer_metrics[
        [
            "SafetyBuffer",
            "SalesMarginYen",
            "LossCostYen",
            "SalesMinusLossCostYen",
            "FulfilledSales",
            "StockoutUnits",
            "SimulatedLoss",
            "ServiceLevel",
            "LossRate",
        ]
    ].style.format(
        {
            "SalesMarginYen": "¥{:,.0f}",
            "LossCostYen": "¥{:,.0f}",
            "SalesMinusLossCostYen": "¥{:,.0f}",
            "FulfilledSales": "{:,.0f}",
            "StockoutUnits": "{:,.0f}",
            "SimulatedLoss": "{:,.0f}",
            "ServiceLevel": "{:.1%}",
            "LossRate": "{:.1%}",
        }
    )
)


In [ ]:
value_plot = (
    fifo_buffer_metrics
    .dropna(subset=["SalesMinusLossCostYen"])
    .sort_values("SafetyBuffer")
    .copy()
)

if not value_plot.empty:
    plt.figure(figsize=(10, 4))
    plt.plot(
        value_plot["SafetyBuffer"],
        value_plot["SalesMinusLossCostYen"],
        marker="o",
    )
    plt.axvline(
        GLOBAL_FIFO_BUFFER,
        linestyle="--",
        label=f"Selected global buffer {GLOBAL_FIFO_BUFFER:+d}",
    )
    plt.title("FIFO sales margin minus loss cost by safety buffer")
    plt.xlabel("Safety buffer")
    plt.ylabel("Sales margin minus loss cost (yen)")
    plt.legend()
    plt.tight_layout()
    plt.show()


### Store–product FIFO policy

Show the buffer selected independently for each store and product when enough validation history exists.

In [ ]:
display(
    selected_fifo_policy.head(30).style.format(
        {
            "EstimatedSalesMarginYen": "¥{:,.0f}",
            "EstimatedLossCostYen": "¥{:,.0f}",
            "EstimatedSalesMinusLossYen": "¥{:,.0f}",
            "ValidationStockoutUnits": "{:,.0f}",
            "ValidationExpiredUnits": "{:,.0f}",
        }
    )
)


### Selected-policy validation summary

Summarize the combined business result of the selected store–product buffers.

In [ ]:
if not selected_policy_summary.empty:
    display(
        selected_policy_summary[
            [
                "Policy",
                "SalesMarginYen",
                "LossCostYen",
                "SalesMinusLossCostYen",
                "ServiceLevel",
                "LossRate",
            ]
        ].style.format(
            {
                "SalesMarginYen": "¥{:,.0f}",
                "LossCostYen": "¥{:,.0f}",
                "SalesMinusLossCostYen": "¥{:,.0f}",
                "ServiceLevel": "{:.1%}",
                "LossRate": "{:.1%}",
            }
        )
    )


## Retrain on all historical data and save

In [ ]:
final_models = {}

for model_name, estimator in model_specs.items():
    pipeline = make_pipeline(
        estimator,
        CATEGORICAL_FEATURES,
        NUMERIC_FEATURES,
    )
    pipeline.fit(model_data[MODEL_FEATURES], model_data[TARGET])
    final_models[model_name] = pipeline
    print("Trained:", model_name)

model_bundle = {
    "models": final_models,
    "ensemble_weights": ENSEMBLE_WEIGHTS,
    "model_features": MODEL_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "lags": LAGS,
    "origin_date": ORIGIN_DATE,
    "last_training_date": model_data["Date"].max(),
    "global_fifo_buffer": GLOBAL_FIFO_BUFFER,
    "selected_fifo_policy": selected_fifo_policy,
    "selected_policy_summary": selected_policy_summary,
    "metrics": metrics,
    "fifo_buffer_metrics": fifo_buffer_metrics,
    "data_mode": DATA_MODE,
}

MODEL_PATH = OUTPUTS_PATH / "demand_model_ensemble_fifo.joblib"
METRICS_PATH = OUTPUTS_PATH / "ensemble_fifo_model_metrics.csv"
OOF_PATH = OUTPUTS_PATH / "ensemble_fifo_oof_predictions.csv"
WEIGHTS_PATH = OUTPUTS_PATH / "ensemble_weights.csv"
FIFO_DETAILS_PATH = OUTPUTS_PATH / "ensemble_fifo_simulation_details.csv"
FIFO_BUFFER_PATH = OUTPUTS_PATH / "ensemble_fifo_buffer_metrics.csv"
FIFO_POLICY_PATH = OUTPUTS_PATH / "ensemble_fifo_selected_policy.csv"
FIFO_POLICY_SUMMARY_PATH = OUTPUTS_PATH / "ensemble_fifo_selected_policy_summary.csv"

joblib.dump(model_bundle, MODEL_PATH)
metrics.to_csv(METRICS_PATH, index=False)
oof_predictions.to_csv(OOF_PATH, index=False)
pd.DataFrame([ENSEMBLE_WEIGHTS]).to_csv(WEIGHTS_PATH, index=False)
fifo_simulation_details.to_csv(FIFO_DETAILS_PATH, index=False)
fifo_buffer_metrics.to_csv(FIFO_BUFFER_PATH, index=False)
selected_fifo_policy.to_csv(FIFO_POLICY_PATH, index=False)
selected_policy_summary.to_csv(FIFO_POLICY_SUMMARY_PATH, index=False)

print("Saved model:", MODEL_PATH)
print("Saved accuracy metrics:", METRICS_PATH)
print("Saved FIFO buffer metrics:", FIFO_BUFFER_PATH)
print("Saved Store–Product FIFO policy:", FIFO_POLICY_PATH)

## Forecast the next sales date

In [ ]:
# FORECAST_DATE is fixed above and is never part of training.
recent_cutoff = TRAINING_END_DATE - pd.Timedelta(days=13)

active_pairs = (
    base_data.loc[
        base_data["Date"].ge(recent_cutoff),
        ["Date", "Store", "Product", "Type"],
    ]
    .sort_values("Date")
    .drop_duplicates(["Store", "Product"], keep="last")
    .drop(columns="Date")
    .reset_index(drop=True)
)

future_rows = active_pairs.copy()
future_rows["Date"] = FORECAST_DATE
future_rows["Demand"] = np.nan

future_calendar = create_calendar(future_rows[["Date"]])
future_rows = future_rows.merge(
    future_calendar,
    on="Date",
    how="left",
    validate="many_to_one",
)

# Weather is unknown for this forecast. Missing values stay internal and are
# handled by the trained preprocessing pipeline; they are not shown below.
for column, value in NUMERIC_DEFAULTS.items():
    if column not in future_rows.columns:
        future_rows[column] = value
for column, value in CATEGORICAL_DEFAULTS.items():
    if column not in future_rows.columns:
        future_rows[column] = value

combined = pd.concat([base_data, future_rows], ignore_index=True, sort=False)
combined_features = build_features(combined, ORIGIN_DATE)
future_features = combined_features.loc[
    combined_features["Date"].eq(FORECAST_DATE)
    & combined_features["Demand"].isna()
].copy()

for column in NUMERIC_FEATURES:
    future_features[column] = pd.to_numeric(
        future_features[column],
        errors="coerce",
    ).astype(float)

for model_name, fitted_model in final_models.items():
    future_features[model_name] = np.clip(
        fitted_model.predict(future_features[MODEL_FEATURES]),
        0,
        None,
    )

future_features["BaseForecastRaw"] = sum(
    ENSEMBLE_WEIGHTS[model_name] * future_features[model_name]
    for model_name in MODEL_NAMES
)
future_features["BaseForecastDemand"] = np.clip(
    np.rint(future_features["BaseForecastRaw"]),
    0,
    None,
).astype(int)

future_features = future_features.merge(
    selected_fifo_policy[["Store", "Product", "SafetyBuffer"]],
    on=["Store", "Product"],
    how="left",
    validate="one_to_one",
)
future_features["SafetyBuffer"] = pd.to_numeric(
    future_features["SafetyBuffer"],
    errors="coerce",
).fillna(GLOBAL_FIFO_BUFFER).astype(int)

future_features["ForecastDemand"] = np.clip(
    np.rint(
        future_features["BaseForecastRaw"]
        + future_features["SafetyBuffer"]
    ),
    0,
    None,
).astype(int)


In [ ]:
opening_carryover = (
    production.loc[
        production["Date"].eq(TRAINING_END_DATE),
        ["Store", "Product", "ClosingStock"],
    ]
    .drop_duplicates(["Store", "Product"], keep="last")
    .rename(columns={"ClosingStock": "OpeningCarryover"})
)

next_day_forecast = future_features[
    [
        "Store",
        "Product",
        "Type",
        "BaseForecastDemand",
        "SafetyBuffer",
        "ForecastDemand",
    ]
].copy()

next_day_forecast = next_day_forecast.merge(
    opening_carryover,
    on=["Store", "Product"],
    how="left",
    validate="one_to_one",
)
next_day_forecast["OpeningCarryover"] = pd.to_numeric(
    next_day_forecast["OpeningCarryover"],
    errors="coerce",
).fillna(0).clip(lower=0)

next_day_forecast["RecommendedProduction"] = np.maximum(
    next_day_forecast["ForecastDemand"]
    - next_day_forecast["OpeningCarryover"],
    0,
).round().astype(int)


In [ ]:
# Actual results are loaded only after every model and FIFO policy has already
# been fitted. They are used for comparison only and cannot leak into training.
actual_columns = [
    "Store",
    "Product",
    "ActualProduction",
    "ActualSales",
    "ActualLoss",
    "ExpectedFIFOLoss",
]
actual_rows = pd.DataFrame(columns=actual_columns)

if EVALUATION_FILE.exists():
    evaluation_data = pd.read_pickle(EVALUATION_FILE)
    evaluation_data["Date"] = pd.to_datetime(
        evaluation_data["Date"]
    ).dt.normalize()

    available_actual_columns = [
        column
        for column in [
            "Date",
            "Store",
            "Product",
            "Production",
            "Sales",
            "Loss",
            "ExpectedLoss",
        ]
        if column in evaluation_data.columns
    ]

    actual_rows = evaluation_data.loc[
        evaluation_data["Date"].eq(FORECAST_DATE),
        available_actual_columns,
    ].copy()
    actual_rows = actual_rows.drop_duplicates(
        ["Store", "Product"],
        keep="last",
    )
    actual_rows = actual_rows.rename(
        columns={
            "Production": "ActualProduction",
            "Sales": "ActualSales",
            "Loss": "ActualLoss",
            "ExpectedLoss": "ExpectedFIFOLoss",
        }
    )

    for column in actual_columns:
        if column not in actual_rows.columns:
            actual_rows[column] = np.nan
    actual_rows = actual_rows[actual_columns]

next_day_forecast = next_day_forecast.merge(
    actual_rows,
    on=["Store", "Product"],
    how="left",
    validate="one_to_one",
)

for column in [
    "ActualProduction",
    "ActualSales",
    "ActualLoss",
    "ExpectedFIFOLoss",
]:
    next_day_forecast[column] = pd.to_numeric(
        next_day_forecast[column],
        errors="coerce",
    )


In [ ]:
def explain_loss(row):
    actual_loss = row["ActualLoss"]
    expected_loss = row["ExpectedFIFOLoss"]
    actual_sales = row["ActualSales"]
    opening_stock = row["OpeningCarryover"]

    if pd.isna(actual_loss):
        return "Actual loss not available"

    if actual_loss == 0:
        if pd.notna(expected_loss) and expected_loss > 0:
            return (
                f"No loss recorded, but FIFO expected {expected_loss:g}; "
                "check stock age or entry"
            )
        return "No loss"

    if pd.isna(expected_loss):
        return "Loss recorded; FIFO reason unavailable"

    if actual_loss == expected_loss:
        return f"{actual_loss:g} old units remained unsold and expired"

    if (
        expected_loss == 0
        and pd.notna(actual_sales)
        and actual_sales >= opening_stock
    ):
        return (
            "Recorded loss despite enough sales to clear opening stock. "
            "Possible causes: FIFO was not followed, products were "
            "damaged during transportation or handling, or the "
            "loss/stock count was entered incorrectly"
        )

    return (
        f"Recorded loss {actual_loss:g} differs from FIFO expected "
        f"{expected_loss:g}; check FIFO or entry"
    )


next_day_forecast["LossReason"] = next_day_forecast.apply(
    explain_loss,
    axis=1,
)

next_day_forecast = next_day_forecast[
    [
        "Store",
        "Product",
        "Type",
        "BaseForecastDemand",
        "SafetyBuffer",
        "ForecastDemand",
        "OpeningCarryover",
        "RecommendedProduction",
        "ActualProduction",
        "ActualSales",
        "ActualLoss",
        "ExpectedFIFOLoss",
        "LossReason",
    ]
].sort_values(["Store", "Product"]).reset_index(drop=True)


### Forecast output

Save and display the final production recommendation together with optional actual-result fields.

In [ ]:
forecast_calendar_row = future_calendar.iloc[0]
date_title = FORECAST_DATE.strftime("%A, %B %d, %Y").replace(" 0", " ")
day_type_title = {
    "平日": "Weekday",
    "週末": "Weekend",
    "祝日": "Holiday",
}.get(str(forecast_calendar_row["DayType"]), str(forecast_calendar_row["DayType"]))

FORECAST_PATH = (
    OUTPUTS_PATH
    / f"{FORECAST_DATE:%Y-%m-%d}_ensemble_fifo_forecast.csv"
)
next_day_forecast.to_csv(FORECAST_PATH, index=False)

display(
    Markdown(
        f"### Forecast and actual results for {date_title} — {day_type_title}"
    )
)
print("Global FIFO fallback buffer:", GLOBAL_FIFO_BUFFER)
print("Saved forecast:", FORECAST_PATH)
display(next_day_forecast)


### Output columns

- `BaseForecastDemand`: weighted ensemble forecast before the FIFO buffer.
- `SafetyBuffer`: selected from FIFO validation, or the global fallback.
- `ForecastDemand`: demand target after the buffer.
- `OpeningCarryover`: observed closing stock from the previous day.
- `RecommendedProduction`: `ForecastDemand - OpeningCarryover`, clipped at zero.
- `ActualProduction`, `ActualSales`, `ActualLoss`, and `ExpectedFIFOLoss`: evaluation-only fields when the separate evaluation file exists.
- `LossReason`: interpretation of recorded loss versus the FIFO expectation.